# Análisis de sentimiento (fase 5)

Notebook de la fase 5. Esta versión tiene la sección **Etiquetado**: arma la muestra de 300 notas (100 por medio), exporta las dos planillas para los anotadores y valida las etiquetas devueltas. Las secciones de evaluación y de evolución se agregan en commits siguientes.

## 1. Etiquetado

> **Importante.** La muestra se saca **solo** del dataset completo y congelado que comparte el equipo (9.261 notas, con los tres medios: MisionesOnline, Infobae y La Nación). La muestra depende de los `id` presentes: si se saca de un `data/noticias_politica.parquet` incompleto, queda distinta. La celda de abajo falla con un mensaje claro si algún medio tiene menos notas elegibles que `PER_MEDIO`.
>
> Este notebook se commitea **sin salidas**. Las planillas contienen texto scrapeado y quedan en `data/etiquetado/` (ignorado por git); solo se commitea `anotaciones/etiquetas.csv` (`id, anotador, etiqueta`, sin texto). La guía para los anotadores está en `docs/guia-etiquetado.md`.

In [ ]:
from pathlib import Path

import pandas as pd

from news_politica.sentiment.labeling import (
    ANOTADORES,
    SAMPLE_SALT,
    export_sheets,
    labeling_sample,
    read_labels,
)

DATA = Path("../data")
ANOTACIONES = Path("../anotaciones")
PER_MEDIO = 100

notes = pd.read_parquet(DATA / "noticias_politica.parquet")
print(len(notes), "notas;", "semilla:", SAMPLE_SALT)
notes["medio"].value_counts()

### Muestra y exportación

`labeling_sample` filtra notas en vivo y títulos vacíos, ordena por `sha1(semilla + id)` dentro de cada medio y toma 100. Los anotadores `a` y `b` se alternan, así que cada uno recibe 50 por medio (150 en total) sin notas en común. `export_sheets` escribe `anotador_a.csv` y `anotador_b.csv` con `id, texto, etiqueta` y sin medio ni salida de modelos.

In [ ]:
sample = labeling_sample(notes, per_medio=PER_MEDIO)
try:
    paths = export_sheets(sample, DATA / "etiquetado")  # never overwrites filled sheets
    print("planillas escritas:", [p.name for p in paths])
except FileExistsError as exc:
    print("No se escribió nada:", exc)
print(sample.groupby(["medio", "anotador"]).size().unstack())
a_ids = set(sample.loc[sample["anotador"] == "a", "id"])
b_ids = set(sample.loc[sample["anotador"] == "b", "id"])
print("ids compartidos:", len(a_ids & b_ids))
print(sample["fuente_texto"].value_counts())

### Unión de etiquetas

Cuando los dos integrantes devuelvan sus archivos completos a `data/etiquetado/`, esta celda los valida con `read_labels` (etiquetas dentro de `pos`, `neu`, `neg`, `descartar`; ids de la muestra; sin repetidos ni vacíos) y escribe `anotaciones/etiquetas.csv` con `id, anotador, etiqueta`.

La celda no depende de la exportación: arma la muestra (determinística) y toma los archivos de `data/etiquetado/` por su nombre. La exportación nunca pisa planillas que ya existen; para regenerarlas hay que pasar `overwrite=True` a propósito.

In [ ]:
sample = labeling_sample(notes, per_medio=PER_MEDIO)
sheet_paths = [DATA / "etiquetado" / f"anotador_{a}.csv" for a in ANOTADORES]
labels = read_labels(sheet_paths, sample)
ANOTACIONES.mkdir(exist_ok=True)
labels.to_csv(ANOTACIONES / "etiquetas.csv", index=False)
print(len(labels), "etiquetas")
labels["etiqueta"].value_counts()